# AI Laboratory: Goal-Based Agents

**Learning Objectives:**
1. Explain the operation of a goal-based intelligent agent
2. Use an LLM as a software engineering assistant
3. Generate, execute, and test LLM-produced code
4. Improve software through iterative prompting
5. Critically evaluate strengths and limitations of LLM-assisted software engineering

**Central Engineering Principle:** Understand → Specify → Generate → Execute → Verify


## Task 1: Understanding the Warehouse Navigation Problem

Before writing any software, we must understand the problem clearly.

### Problem Specification

**Environment:**
- A warehouse represented as a 2D grid
- S = starting position (row 1, col 1)
- G = goal position (row 1, col 20)
- '#' = obstacles (shelving units)
- '.' = free space

**Agent Actions:**
- Move Up
- Move Down
- Move Left
- Move Right

Each move changes the agent's position by one grid square. Movement is invalid if it would hit an obstacle or go out of bounds.

**Goal:** Find a collision-free path from S to G.

**Why is this a goal-based agent?**
The agent has an explicit goal state (position of G) and uses internal planning/search to determine which sequence of actions will achieve that goal. Unlike a reflex agent that reacts only to immediate stimuli, this agent maintains a model of the environment and searches for a path.

In [ ]:
# Define the warehouse map
warehouse_map = """
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
"""

print("Warehouse Layout:")
print(warehouse_map)

# Parse the warehouse
lines = warehouse_map.strip().split('\n')
warehouse_grid = [list(line) for line in lines]

# Find start and goal positions
start_pos = None
goal_pos = None

for i in range(len(warehouse_grid)):
    for j in range(len(warehouse_grid[i])):
        if warehouse_grid[i][j] == 'S':
            start_pos = (i, j)
        elif warehouse_grid[i][j] == 'G':
            goal_pos = (i, j)

print(f"\nStart position: {start_pos}")
print(f"Goal position: {goal_pos}")
print(f"Grid dimensions: {len(warehouse_grid)} x {len(warehouse_grid[0])}")

## Task 2: Designing the Agent Architecture

Before asking an LLM, we specify the agent design:

```
┌─────────────────────────────────────────┐
│         GOAL-BASED AGENT LOOP           │
├─────────────────────────────────────────┤
│  1. Environment State                   │
│     └─> Current position (x, y)         │
│                                         │
│  2. Goal                                │
│     └─> Reach position of 'G'           │
│                                         │
│  3. Search Component (BFS/A*)           │
│     └─> Find sequence of actions        │
│                                         │
│  4. Execute Path                        │
│     └─> Follow the path step by step    │
│                                         │
│  5. Validation                          │
│     └─> Verify path avoids obstacles    │
└─────────────────────────────────────────┘
```

### Design Decisions:

1. **State Representation:** (row, col) tuple
2. **Warehouse Representation:** 2D list of characters
3. **Valid Actions:** Check grid boundaries and obstacles
4. **Goal Test:** Check if current position == goal position
5. **Frontier:** Use a queue (BFS) or priority queue (A*)
6. **Path Reconstruction:** Store parent pointers during search
7. **Output:** Path as list of positions, path length, states explored

## Task 3: LLM Prompt Engineering

### Prompt Given to Claude:

```
Write a well-documented Python program implementing a goal-based agent for warehouse navigation.

Warehouse map:
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################

Requirements:
- Represent the warehouse as a 2D grid
- S = start, G = goal, # = obstacle, . = free cell
- Agent can move Up, Down, Left, Right (one cell per move)
- Find a collision-free path from S to G using BFS or A*
- Print the path found or "No path exists" if impossible
- Report: path length, number of states explored
- Include validation that path avoids all obstacles

Keep it simple, well-commented, and testable.
```

### Response Analysis:
The LLM typically:
- Chooses BFS (simpler) or A* (more sophisticated)
- Uses a queue or priority queue
- Implements parent tracking for path reconstruction
- Validates moves against grid boundaries and obstacles

## Manually Implemented Solution (Reference Implementation)

To validate LLM-generated code, we implement a trusted version ourselves.

In [ ]:
from collections import deque
import heapq

class WarehouseAgent:
    """Goal-based agent for warehouse navigation."""
    
    def __init__(self, warehouse_map_str):
        """Initialize agent with warehouse map."""
        lines = warehouse_map_str.strip().split('\n')
        self.grid = [list(line) for line in lines]
        self.rows = len(self.grid)
        self.cols = len(self.grid[0])
        
        # Find start and goal
        self.start = None
        self.goal = None
        for i in range(self.rows):
            for j in range(self.cols):
                if self.grid[i][j] == 'S':
                    self.start = (i, j)
                elif self.grid[i][j] == 'G':
                    self.goal = (i, j)
        
        self.states_explored = 0
    
    def is_valid_move(self, pos):
        """Check if a position is valid (in bounds and not obstacle)."""
        row, col = pos
        if row < 0 or row >= self.rows or col < 0 or col >= self.cols:
            return False
        return self.grid[row][col] != '#'
    
    def get_neighbors(self, pos):
        """Get valid neighboring positions (up, down, left, right)."""
        row, col = pos
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
        neighbors = []
        for dr, dc in directions:
            new_pos = (row + dr, col + dc)
            if self.is_valid_move(new_pos):
                neighbors.append(new_pos)
        return neighbors
    
    def bfs_search(self):
        """Breadth-First Search for shortest path."""
        if not self.start or not self.goal:
            return None, 0
        
        self.states_explored = 0
        queue = deque([self.start])
        visited = {self.start}
        parent = {self.start: None}
        
        while queue:
            current = queue.popleft()
            self.states_explored += 1
            
            if current == self.goal:
                # Reconstruct path
                path = []
                node = current
                while node is not None:
                    path.append(node)
                    node = parent[node]
                return path[::-1], len(path) - 1  # Return path and length
            
            for neighbor in self.get_neighbors(current):
                if neighbor not in visited:
                    visited.add(neighbor)
                    parent[neighbor] = current
                    queue.append(neighbor)
        
        return None, 0  # No path found
    
    def solve_bfs(self):
        """Solve using BFS."""
        path, length = self.bfs_search()
        return {
            'algorithm': 'BFS',
            'path': path,
            'path_length': length,
            'states_explored': self.states_explored,
            'solution_found': path is not None
        }
    
    def format_result(self, result):
        """Format and display result."""
        print(f"\nAlgorithm: {result['algorithm']}")
        print(f"Solution Found: {result['solution_found']}")
        
        if result['solution_found']:
            print(f"Path Length: {result['path_length']}")
            print(f"Path: {result['path'][:5]}..." if len(result['path']) > 5 else f"Path: {result['path']}")
        
        print(f"States Explored: {result['states_explored']}")
        return result

# Test the reference implementation
agent = WarehouseAgent(warehouse_map)
result = agent.solve_bfs()
result = agent.format_result(result)

print(f"\n✓ Reference implementation successful")

## Task 4: Test the Agent with Multiple Scenarios

We test with:
1. **Test A**: Original warehouse (solvable)
2. **Test B**: Trivial case (goal adjacent to start)
3. **Test C**: Impossible case (goal unreachable)
4. **Test D**: Alternative paths (check optimality)

In [ ]:
# TEST A: Original warehouse (solvable)
print("="*60)
print("TEST A: Original Warehouse (Solvable)")
print("="*60)

test_a_result = agent.solve_bfs()
agent.format_result(test_a_result)

# Validate path
if test_a_result['solution_found']:
    path = test_a_result['path']
    print(f"\nPath Validation:")
    print(f"  - Start position: {path[0]}")
    print(f"  - Goal position: {path[-1]}")
    print(f"  - All positions valid: ", end="")
    
    valid = all(agent.is_valid_move(pos) for pos in path)
    print("✓ YES" if valid else "✗ NO")
    
    # Check consecutive moves are adjacent
    adjacent = True
    for i in range(len(path)-1):
        r1, c1 = path[i]
        r2, c2 = path[i+1]
        if abs(r1-r2) + abs(c1-c2) != 1:
            adjacent = False
            break
    print(f"  - All moves are adjacent: {'✓ YES' if adjacent else '✗ NO'}")

In [ ]:
# TEST B: Trivial case - goal adjacent to start
print("\n" + "="*60)
print("TEST B: Trivial Case (Goal Adjacent)")
print("="*60)

trivial_map = """
#####
#SG##
#####
"""

agent_b = WarehouseAgent(trivial_map)
result_b = agent_b.solve_bfs()
agent_b.format_result(result_b)

assert result_b['solution_found'], "TEST B FAILED: Should find solution"
assert result_b['path_length'] == 1, "TEST B FAILED: Path length should be 1"
print("✓ TEST B PASSED")

In [ ]:
# TEST C: Impossible case - goal unreachable
print("\n" + "="*60)
print("TEST C: Impossible Case (Unreachable Goal)")
print("="*60)

impossible_map = """
#######
#S....#
###.###
#...#G#
#######
"""

agent_c = WarehouseAgent(impossible_map)
result_c = agent_c.solve_bfs()
agent_c.format_result(result_c)

assert not result_c['solution_found'], "TEST C FAILED: Should not find solution"
print("✓ TEST C PASSED")

In [ ]:
# TEST D: Multiple paths - check for shortest path
print("\n" + "="*60)
print("TEST D: Multiple Paths (Shortest Path)")
print("="*60)

multi_path_map = """
#########
#S.....G#
#.#####.#
#.......#
#########
"""

agent_d = WarehouseAgent(multi_path_map)
result_d = agent_d.solve_bfs()
agent_d.format_result(result_d)

# BFS guarantees shortest path in unweighted graph
assert result_d['solution_found'], "TEST D FAILED: Should find solution"
print(f"\n✓ BFS guarantees shortest path (length={result_d['path_length']})")
print(f"  - Direct distance (Manhattan): {abs(agent_d.goal[0]-agent_d.start[0]) + abs(agent_d.goal[1]-agent_d.start[1])}")
print(f"  - Actual path length: {result_d['path_length']}")

## Task 5: Reflection on LLM-Assisted Development

### Questions and Answers:

**1. Did the LLM generate a working program on the first attempt?**

Typical answer: The LLM usually generates code that runs without syntax errors, but may have logic errors or inefficiencies. Common issues:
- Incorrect path reconstruction (reversed path)
- Missing goal check
- Using wrong data structures

**2. If not, how can we improve the prompt?**

Key improvements:
- Be specific about data structures (use "deque" not "queue")
- Specify the exact algorithm ("Breadth-First Search" not "search")
- Include validation requirements
- Request clear output format
- Ask for explanation of key components

**3. What search algorithm did the LLM choose?**

Most often: BFS (simplicity) or A* (if heuristic suggested). Both are reasonable, but:
- BFS: Guarantees shortest path, simpler to implement
- A*: More efficient with good heuristic

**4. Why did the LLM select this algorithm?**

The LLM likely chose based on:
- Specification suggested BFS in comments
- Simplicity and reliability
- No need for domain-specific heuristics
- Training data prevalence

## Task 6: What We Learned

### About Goal-Based Agents:

1. **Goal Representation**: Explicit goal state enables systematic search
2. **State Space Search**: Exploring states until goal is found
3. **Path Planning**: Finding sequence of actions, not just next action
4. **Optimality**: BFS finds shortest path; A* with admissible heuristic also optimal

### About LLM-Assisted Development:

1. **Specification is Critical**: Precise requirements lead to better code
2. **Testing is Non-Negotiable**: Generated code must be validated
3. **Engineer Remains Responsible**: LLM is a tool, not an oracle
4. **Iterative Refinement**: Multiple prompts often better than one

### Engineering Principle Demonstrated:

```
UNDERSTAND
    ↓
SPECIFY (clear requirements)
    ↓
GENERATE (LLM produces code)
    ↓
EXECUTE (run the code)
    ↓
VERIFY (test multiple scenarios)
    ↓
If OK: Deploy
If not: Refine specification → GENERATE again
```

## Summary

✓ Explained goal-based agent operation  
✓ Designed agent architecture before coding  
✓ Specified clear requirements for LLM  
✓ Implemented reference implementation  
✓ Tested with 4 diverse scenarios  
✓ Validated correctness (path existence, adjacency, obstacles)  
✓ Reflected on LLM effectiveness and limitations  

**Key Takeaway**: An LLM can accelerate implementation, but engineering discipline—specification, testing, validation—remains essential.